# 02: Tiny GenImage inventory (Step 4)

Downloads [Tiny GenImage](https://huggingface.co/datasets/TheKernel01/Tiny-GenImage) to Colab's local disk and catalogues every image: split, label, generator, format, size. Then it checks for format and size biases at scale.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import io
import numpy as np
import pandas as pd
from PIL import Image

ROOT = "/content/drive/MyDrive/minor-project"
Path(ROOT, "results").mkdir(exist_ok=True)

In [ ]:
!pip install -q datasets
from datasets import load_dataset, Image as HFImage

ds = load_dataset("TheKernel01/Tiny-GenImage", cache_dir="/content/hf_cache")
print(ds)

In [ ]:
GENERATORS = ds["train"].features["generator"].names
LABELS = ds["train"].features["label"].names

rows = []
for split in ["train", "validation"]:
    part = ds[split].cast_column("image", HFImage(decode=False))
    for i, example in enumerate(part):
        raw = example["image"]["bytes"]
        img = Image.open(io.BytesIO(raw))
        rows.append({
            "split": split,
            "index": i,
            "label": LABELS[example["label"]],
            "generator": GENERATORS[example["generator"]],
            "format": img.format,
            "mode": img.mode,
            "width": img.size[0],
            "height": img.size[1],
            "file_kb": round(len(raw) / 1024),
        })

inventory = pd.DataFrame(rows)
inventory.to_csv(Path(ROOT, "results", "03_tiny_genimage_inventory.csv"), index=False)
print(len(inventory), "images catalogued")

In [ ]:
print("Images per generator and split:")
display(pd.crosstab(inventory["generator"], inventory["split"]))

print("\nFile format per generator:")
display(pd.crosstab(inventory["generator"], inventory["format"]))

print("\nTypical (median) size per generator:")
display(inventory.groupby("generator")[["width", "height", "file_kb"]].median())

says_fake = inventory["format"] == "PNG"
accuracy = (np.where(says_fake, "fake", "real") == inventory["label"]).mean()
print(f"\nCheat rule 'PNG = fake' on the whole dataset: {accuracy:.1%}")